# 05 — The sbot Buffering Bug: A Case Study

**Goal:** Reproduce the exact bug that happened in sbot, understand why, and implement the fix.

## The bug

sbot's agent processes a user message through multiple steps:
1. Emit "thinking..." → user should see this immediately
2. Call LLM (takes seconds)
3. Emit "tool call: read_file(...)" → user should see this
4. Execute tool
5. Emit "tool result: ..." → user should see this
6. Emit final response

**What actually happened:** Steps 1, 3, 5, 6 all appeared at the SAME TIME — like a dump of buffered output. The user waited minutes seeing nothing, then everything appeared at once.

## Exercise 5.1 — Reproduce the bug (async queue version)

This simulates the original sbot architecture with an async outbound queue.

In [ ]:
import asyncio
import time

outbound_queue = asyncio.Queue()

async def display_loop():
    """Consumer: displays messages as they arrive."""
    while True:
        msg = await outbound_queue.get()
        print(f"  [{time.time():.2f}] DISPLAY: {msg}")

async def agent_buggy():
    """Producer: emits events while processing a user message."""
    # Step 1: thinking
    outbound_queue.put_nowait("⟳ thinking...")         # NO await — no yield
    
    # Step 2: LLM call
    await asyncio.sleep(2)  # simulates LLM API call
    
    # Step 3: emit multiple events rapidly (NO yields between them)
    outbound_queue.put_nowait("💭 I think we should...")  # NO await
    outbound_queue.put_nowait("📊 tokens: 150")           # NO await
    outbound_queue.put_nowait("🔧 read_file(main.py)")    # NO await
    
    # Step 4: tool execution (blocking — but let's use sleep)
    await asyncio.sleep(0.5)  # simulates tool
    
    # Step 5: more rapid emits
    outbound_queue.put_nowait("→ result: 50 lines")       # NO await
    outbound_queue.put_nowait("✅ Here's your answer...")  # NO await

display_task = asyncio.create_task(display_loop())
await agent_buggy()
await asyncio.sleep(0.5)  # let display catch up
display_task.cancel()

### Question 5.1
Look at the timestamps. Did the display show messages one by one? Or did some batch together?

Which messages batched? Why those specific ones?

*Your answer:*



## Exercise 5.2 — Failed fix: sleep(0) between emits

In [ ]:
async def agent_sleep0():
    outbound_queue.put_nowait("⟳ thinking...")
    await asyncio.sleep(0)  # yield
    
    await asyncio.sleep(2)  # LLM
    
    outbound_queue.put_nowait("💭 I think we should...")
    await asyncio.sleep(0)  # yield
    outbound_queue.put_nowait("📊 tokens: 150")
    await asyncio.sleep(0)  # yield
    outbound_queue.put_nowait("🔧 read_file(main.py)")
    await asyncio.sleep(0)  # yield
    
    await asyncio.sleep(0.5)  # tool
    
    outbound_queue.put_nowait("→ result: 50 lines")
    await asyncio.sleep(0)  # yield
    outbound_queue.put_nowait("✅ Here's your answer...")
    await asyncio.sleep(0)

outbound_queue = asyncio.Queue()  # fresh queue
display_task = asyncio.create_task(display_loop())
await agent_sleep0()
await asyncio.sleep(0.5)
display_task.cancel()

### Question 5.2
Better? Worse? Same? Does `sleep(0)` guarantee the display runs after each emit?

**Hint:** `sleep(0)` puts you at the END of the ready queue. But if the display task just woke up from `queue.get()`, is it already in the ready queue or not?

*Your answer:*



## Exercise 5.3 — The REAL fix: sync callbacks

Instead of putting messages in a queue for another task to display, call the display function DIRECTLY — in the same call stack, no scheduling needed.

In [ ]:
def display_sync(msg):
    """Sync callback — runs IMMEDIATELY in the caller's stack."""
    print(f"  [{time.time():.2f}] DISPLAY: {msg}", flush=True)

async def agent_fixed():
    display_sync("⟳ thinking...")           # runs NOW
    
    await asyncio.sleep(2)                    # LLM
    
    display_sync("💭 I think we should...")  # runs NOW
    display_sync("📊 tokens: 150")           # runs NOW
    display_sync("🔧 read_file(main.py)")    # runs NOW
    
    await asyncio.sleep(0.5)                  # tool
    
    display_sync("→ result: 50 lines")       # runs NOW
    display_sync("✅ Here's your answer...")  # runs NOW

await agent_fixed()

### Question 5.3
Now every message appeared instantly. But wait — if the display is a sync function call, how does this work for Telegram (which needs an async HTTP POST to deliver messages)?

*Your answer:*



## Exercise 5.4 — Sync callback + async delivery (the sbot pattern)

For network channels: the sync callback just QUEUES the message (instant), a background task does the actual HTTP POST. Then we yield slightly to let that background task run.

In [ ]:
telegram_queue = asyncio.Queue()

def telegram_callback(msg):
    """Sync callback: just queues. Instant, non-blocking."""
    telegram_queue.put_nowait(msg)

async def telegram_sender():
    """Background task: does the actual 'HTTP POST' (simulated)."""
    while True:
        msg = await telegram_queue.get()
        await asyncio.sleep(0.01)  # simulate HTTP POST
        print(f"  [{time.time():.2f}] TELEGRAM SENT: {msg}")

async def emit(msg):
    """Emit to all channels."""
    display_sync(msg)          # CLI: immediate
    telegram_callback(msg)     # Telegram: queued
    await asyncio.sleep(0.05)  # yield so telegram_sender can POST

sender = asyncio.create_task(telegram_sender())

print("=== Agent processing ===")
await emit("⟳ thinking...")
await asyncio.sleep(1)  # LLM
await emit("🔧 tool call")
await asyncio.sleep(0.3)  # tool
await emit("✅ response")

await asyncio.sleep(0.2)
sender.cancel()

### Question 5.4
Both CLI (sync) and Telegram (async) got messages one by one. Why does `sleep(0.05)` work for Telegram but `sleep(0)` didn't work earlier?

*Your answer:*



## Summary: The sbot buffering bug

```
ROOT CAUSE: Between two await points, no other task can run.
            If you emit 5 messages without yielding, consumers see all 5 at once.

FIX FOR CLI:     Sync callback → print() runs in the same call stack. Instant.
FIX FOR NETWORK: Sync callback queues, yield 50ms for sender to POST.
FUTURE FIX:      External message broker (Redis) — producer and consumer
                 are in different processes, no scheduling dependency.
```

**Next notebook:** Redis Streams — the external message broker approach